In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
import neat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.special import softmax

In [2]:
SEED = 42

np.random.seed(SEED)

In [3]:
print(f"Cargando datos...")
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)


Cargando datos...


In [4]:
print(dataset_train.klass.value_counts())

klass
neutral     1485
positive     968
negative     689
Name: count, dtype: int64


In [5]:
le = LabelEncoder()
X_trainext = dataset_train['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es = np.vstack(dataset_train["we_es"].to_numpy())
X_mx = np.vstack(dataset_train["we_mx"].to_numpy())
X_ft = np.vstack(dataset_train["we_ft"].to_numpy())
Y_train = dataset_train['klass'].to_numpy()

X_train = np.concatenate([X_es, X_mx, X_ft], axis=1) 

X_test_text = dataset_test['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es_t = np.vstack(dataset_test["we_es"].to_numpy())
X_mx_t = np.vstack(dataset_test["we_mx"].to_numpy())
X_ft_t = np.vstack(dataset_test["we_ft"].to_numpy())
X_test = np.concatenate([X_es_t, X_mx_t, X_ft_t], axis=1) 
Y_test = dataset_test['klass'].to_numpy()

In [6]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Y_train_encoded= le.fit_transform(Y_train)
Y_test_encoded= le.transform(Y_test)


In [7]:
total_features = X_train.shape[1] # 900
n_comp = int(total_features * 0.30)
pca = PCA(n_components=n_comp)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


In [8]:
X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
    X_train,
    Y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=Y_train_encoded
)

In [9]:
input_size = X_train.shape[1]
input_size

270

In [10]:
def eval_genomes(genomes, config):

    BATCH_SIZE = 128

    indices = np.random.choice(len(X_train), BATCH_SIZE, replace=False)

    X_batch = X_train[indices]
    Y_batch = Y_train_encoded[indices]

    for genome_id, genome in genomes:

        net = neat.nn.FeedForwardNetwork.create(genome, config)

        predictions = []

        for xi in X_batch:
            output = net.activate(xi)
            pred = np.argmax(output)
            predictions.append(pred)

        fitness = f1_score(
            Y_batch,
            predictions,
            average="macro"
        )
        complexity_penalty = len(genome.connections) * 0.0005

        genome.fitness = fitness - complexity_penalty

In [11]:
def run_neat(config_path):
    # Cargar configuración
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction,
                         neat.DefaultSpeciesSet, neat.DefaultStagnation,
                         config_path)

    # Crear la población
    p = neat.Population(config)

    # Agregar reporteros para ver el progreso en consola
    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    # Ejecutar por N generaciones
    winner = p.run(eval_genomes, 100) 
    
    return winner, config


In [20]:

# Ejecución
winner_genome, config_used = run_neat('config-feedforward.cfg')


 ****** Running generation 0 ****** 

Population's average fitness: 0.22835 stdev: 0.04634
Best fitness: 0.34894 - size: (3, 162) - species 1 - id 357
Average adjusted fitness: 0.120
Mean genetic distance 2.256, standard deviation 0.228
Population of 400 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0   400      0.349    0.120     0
Total extinctions: 0
Generation time: 3.369 sec

 ****** Running generation 1 ****** 

Population's average fitness: 0.26166 stdev: 0.05548
Best fitness: 0.44354 - size: (3, 162) - species 1 - id 763
Average adjusted fitness: 0.140
Mean genetic distance 2.329, standard deviation 0.311
Population of 400 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1   400      0.444    0.140     0
Total extinctions: 0
Generation time: 3.806 sec (3.588 average)

 ****** Running generatio

In [21]:
# Reconstruir la mejor red
winner_net = neat.nn.FeedForwardNetwork.create(winner_genome, config_used)

# Predecir en el conjunto de TEST
test_predictions = []

for xi in X_test:

    output = winner_net.activate(xi)

    probs = softmax(output)

    pred = np.argmax(probs)

    test_predictions.append(pred)

# Calcular métrica final para el reporte
final_score = f1_score(Y_test_encoded, test_predictions, average="macro")
print(f"F1-Score de la mejor red NEAT en Test: {final_score}")

F1-Score de la mejor red NEAT en Test: 0.38830349471132647


In [22]:
from sklearn.metrics import classification_report
# TODO: Evaluar el desempeño del modelo 
print(classification_report(Y_test_encoded, test_predictions, digits=4,target_names=['negative', 'neutral', 'positive']))

              precision    recall  f1-score   support

    negative     0.2385    0.1792    0.2046       173
     neutral     0.5163    0.5984    0.5543       371
    positive     0.4204    0.3926    0.4060       242

    accuracy                         0.4427       786
   macro avg     0.3917    0.3900    0.3883       786
weighted avg     0.4256    0.4427    0.4317       786

